# EDA – Erste Auswertung der Zwillinge (Stufe 1 und 2)

**Autorin:** Nadia Litzenburger · *KI-Nutzung: Code-Entwürfe mit Claude (Anthropic); Fragestellung und Bewertung bei der Autorin.*

**Daten:** 600 Zwillinge aus Aylas Vollauf (120 Zwillings-Quellen × 5 Varianten), Markt aus
`jobmarket_data-engineer_180cv_2026-10-20.parquet`. Jeder Zwilling wird mit **seinem Original** verglichen, beide gegen
denselben Job, jeweils einzeln in dieselbe Bewerberliste (60 CVs) einsortiert.

| Variante | geändert | Richtungen |
|---|---|---|
| edu | Bildungsweg | academic, certification, self_taught |
| lin | Laufbahn | linear, non_linear |
| tone | Ton | confident, hedged |
| para | nur umformuliert, kein Merkmal → **Rauschgrenze** | – |
| all | edu + lin + tone zusammen | benachteiligt |

**Methoden:** MeritMe mit reinem ESCOX und mit Hybrid (Wortsuche ∪ ESCOX ∩ IT-Skills, siehe `eda_escox_extraktion.ipynb`),
in der Version v2 mit Tobis Zerlegung (`escox_v2_it`, `hybrid_v2`) und ohne Studienfach-Skills (`hybrid_v2_noedu`),
dazu eine **vorläufige** TF-IDF-Baseline (maßgeblich ist Aylas TF-IDF-Pipeline). Das Notebook wertet automatisch
alle Methoden aus, die `evaluate_twins.py` gerechnet hat.

**Fragen:**
1. Bleiben die ESCO-Labels stabil? (Stufe 1, Ziel Ø Jaccard ≥ 0,85)
2. Welche Labels lösen die Proxies aus, und betreffen sie den Job?
3. Wie weit verschiebt sich der Rang, und ist das mehr als beim reinen Umformulieren? (Stufe 2)
4. In welche **Richtung**: Werden bestimmte Merkmale systematisch bevorzugt?

**Voraussetzung:** `python scripts/evaluate_twins.py --twins data/processed/vollauf_zwillinge.html` wurde einmal ausgeführt
(erzeugt `twin_eval.parquet` und den Label-Cache).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "labels.py").exists())
sys.path.append(str(ROOT / "scripts"))
from labels import label_market, label_names, load_market
from evaluate_twins import load_twins, rank_biserial

TWINS_PATH = ROOT / "data/processed/vollauf_zwillinge.html"
EVAL_PATH = ROOT / "data/processed/twin_eval.parquet"
JACCARD_ZIEL, ALPHA = 0.85, 0.05

INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#c3c2b7"
METHOD_COLORS = {"escox": GREY, "escox_it": "#a8a79d", "hybrid": "#86b6ef", "escox_v2_it": "#7a7970",
                 "hybrid_v2": AQUA, "hybrid_v2_noedu": "#0f7a55", "tfidf": BLUE}
VARIANTS = ["para", "edu", "lin", "tone", "all"]
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.spines.left": False, "axes.grid": True,
                     "grid.color": GRID, "font.size": 10})

market = load_market()
twins = load_twins(TWINS_PATH, market)
twins["richtung"] = twins["richtung"].replace("", "–")      # para hat keine Richtung
res = pd.read_parquet(EVAL_PATH)
res["richtung"] = res["richtung"].replace("", "–")
res = res.merge(twins[["twin_id", "cv_typ", "laengenverhaeltnis", "laenge_flag"]], on="twin_id", how="left")
# Vorzeichen: positiv = Zwilling steht besser da als sein Original
res["delta_score"] = res["score_twin"] - res["score_orig"]
res["rang_gewinn"] = res["rank_a"] - res["rank_b"]          # rank_a = Original, rank_b = Zwilling
main = res[res["test"] == "haupttest"]
# alle MeritMe-Methoden, die evaluate_twins.py gerechnet hat, in fester Reihenfolge
MERIT = [m for m in ["escox", "escox_it", "hybrid", "escox_v2_it", "hybrid_v2", "hybrid_v2_noedu"]
         if m in set(res["methode"])]
HAS_TFIDF = "tfidf" in set(res["methode"])          # Baseline nur, wenn mit --tfidf gerechnet
ALL_M = MERIT + (["tfidf"] if HAS_TFIDF else [])
bar_offsets = lambda n: (np.arange(n) - (n - 1) / 2) * (0.8 / n)   # Balken nebeneinander, Breite 0,8
print(f"{twins['twin_id'].nunique()} Zwillinge | Ergebniszeilen: {len(res)} | Methoden: {ALL_M}")

## 1. Überblick über die Zwillinge

In [ ]:
display(pd.crosstab(twins["variante"], twins["ring"], margins=True).reindex(VARIANTS + ["All"]))
display(twins.groupby("variante")["richtung"].value_counts().unstack(fill_value=0).reindex(VARIANTS))

**Länge:** Die Zahl der ESCO-Labels hängt an der Textlänge. Ändert ein Zwilling die Länge stark, kann das die Labels
verschieben, ohne dass sich die Qualifikation ändert.

In [ ]:
display(twins.groupby("variante").agg(laenge_median=("laengenverhaeltnis", "median"),
                                      laenge_max=("laengenverhaeltnis", "max"),
                                      laengen_flag=("laenge_flag", "sum"),
                                      aenderungsanteil=("aenderungsanteil", "median")).reindex(VARIANTS).round(2))

## 2. Stufe 1: Bleiben die Labels stabil?

Jaccard zwischen den Label-Mengen von Original und Zwilling (1 = identisch). Nur Haupttest (Ring A + B).

In [ ]:
s1 = main[main["methode"] != "tfidf"]
stufe1 = (s1.groupby(["variante", "methode"])["jaccard"]
            .agg(Mittel="mean", Anteil_ab_085=lambda j: (j >= JACCARD_ZIEL).mean())
            .unstack().reindex(VARIANTS).round(2))
stufe1

In [ ]:
mean = s1.pivot_table(index="variante", columns="methode", values="jaccard", aggfunc="mean").reindex(VARIANTS)
fig, ax = plt.subplots(figsize=(9, 3.8))
x, off = np.arange(len(mean)), bar_offsets(len(MERIT))
for i, m in enumerate(MERIT):
    ax.bar(x + off[i], mean[m], width=0.8 / len(MERIT), color=METHOD_COLORS[m], label=m)
ax.axhline(JACCARD_ZIEL, color=ORANGE, lw=1, ls="--")
ax.annotate(f"Ziel Stufe 1: {JACCARD_ZIEL}", (x[0] - 0.45, JACCARD_ZIEL), xytext=(0, 4),
            textcoords="offset points", fontsize=8, color=ORANGE)
ax.set_xticks(x, mean.index); ax.set_ylim(0, 1.05); ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)
ax.set_ylabel("Ø Jaccard Original ↔ Zwilling")
ax.set_title("Stufe 1: Label-Stabilität je Variante (Haupttest)", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Nach Ring und CV-Typ:** Ring B (Python) hat die wenigsten Labels, dort wiegt jede Änderung schwerer. Der CV-Typ
(Liste, Prosa, kurz) zeigt, ob die Formatierung eine Rolle spielt (vgl. `eda_escox_extraktion.ipynb`).

In [ ]:
display(s1.pivot_table(index="ring", columns="methode", values="jaccard", aggfunc="mean").round(2))
s1.pivot_table(index="cv_typ", columns=["methode"], values="jaccard", aggfunc="mean").round(2)

## 3. Welche Labels lösen die Proxies aus?

Labels, die im Zwilling **neu** sind (nicht im Original). Entscheidend ist, ob sie im **Job** stehen: Nur dann ändern
sie den Score. Sonst ändern sie nur die Extra-Liste im Dashboard.

In [ ]:
def labels_for(method):
    m = label_market(market, method)
    t = label_market(twins[["twin_id", "text_zwilling"]].rename(columns={"twin_id": "id", "text_zwilling": "text"}), method)
    return dict(zip(m["id"], m["labels"])), dict(zip(t["id"], t["labels"]))

rows = []
for method in MERIT:
    lab, tlab = labels_for(method)
    for _, t in twins.iterrows():
        orig, twin, job = lab[t["source_id"]], tlab[t["twin_id"]], lab[t["assigned_job_id"]]
        for uri in twin - orig:
            rows.append({"methode": method, "variante": t["variante"], "richtung": t["richtung"],
                         "label": label_names([uri])[0], "im_job": uri in job})
neu = pd.DataFrame(rows)
top = (neu.groupby(["methode", "variante", "label"]).agg(anzahl=("label", "size"), im_job=("im_job", "sum"))
          .reset_index().sort_values("anzahl", ascending=False).groupby(["methode", "variante"]).head(3))
top.set_index(["methode", "variante"]).sort_index()

In [ ]:
print("Anteil neuer Labels, die auch im zugeordneten Job stehen (nur diese ändern den Score):")
neu.groupby(["methode", "variante"])["im_job"].agg(neue_labels="size", davon_im_job="sum").unstack(0).reindex(VARIANTS)

## 4. Stufe 2: Wie weit verschiebt sich der Rang?

Ø Rangabstand zwischen Original und Zwilling (ohne Vorzeichen). **„para“ ist die Rauschgrenze**: Ein Proxy-Effekt zählt
nur, wenn er deutlich darüber liegt. Spalte `über_para` = Rangabstand der Variante minus Rangabstand bei para.

In [ ]:
dist = main.pivot_table(index="variante", columns="methode", values="distance", aggfunc="mean").reindex(VARIANTS)[ALL_M]
ueber = (dist - dist.loc["para"]).add_suffix("_über_para")
pd.concat([dist, ueber], axis=1).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))
x, off = np.arange(len(dist)), bar_offsets(len(ALL_M))
for i, m in enumerate(ALL_M):
    ax.bar(x + off[i], dist[m], width=0.8 / len(ALL_M), color=METHOD_COLORS[m],
           label=m + (" (vorläufig)" if m == "tfidf" else ""))
ax.set_xticks(x, dist.index); ax.set_ylabel("Ø Rangabstand (Plätze)")
ax.set_title("Stufe 2: Rangabstand Original ↔ Zwilling (para = Rauschgrenze)", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Abstand 0 und Gleichstand:** Der MeritMe-Score ist grob. Haben Original und Zwilling denselben Score, ist der Abstand 0.
Das ist echte Gleichbehandlung, wenn die Labels gleich sind, aber nur Grobkörnigkeit, wenn sich die Labels ändern und
der Score trotzdem gleich bleibt.

In [ ]:
g = main[main["methode"] != "tfidf"].copy()
g["fall"] = np.select([g["distance"] > 0, g["jaccard"] == 1], ["Rang verschoben", "Abstand 0, Labels gleich"],
                      "Abstand 0, Labels verschieden")
pd.crosstab([g["methode"], g["variante"]], g["fall"], normalize="index").round(2)

**Wilcoxon-Test** je Variante und Methode (nur mit Baseline, also `--tfidf` bzw. Aylas TF-IDF): Ist der Rangabstand bei MeritMe kleiner als bei TF-IDF (einseitig, gepaart)?
Effektstärke r = rank-biserial (negativ = MeritMe stabiler; Erfolgskriterium |r| ≥ 0,3). Mit 10 Tests: Bonferroni-Schwelle.

In [ ]:
tests = []
if not HAS_TFIDF:
    print("Keine Baseline in twin_eval.parquet: evaluate_twins.py mit --tfidf ausführen bzw. Aylas TF-IDF anbinden.")
for v, gr in (main.groupby("variante") if HAS_TFIDF else []):
    wide = gr.pivot(index="twin_id", columns="methode", values="distance")
    for m in MERIT:
        d = wide[m] - wide["tfidf"]
        p = wilcoxon(d, alternative="less").pvalue if (d != 0).any() else np.nan
        tests.append({"variante": v, "methode": m, "n": len(d), "MeritMe_kleiner": (d < 0).mean(),
                      "gleich": (d == 0).mean(), "MeritMe_größer": (d > 0).mean(), "p": p, "r": rank_biserial(d)})
tests = pd.DataFrame(tests, columns=["variante", "methode", "n", "MeritMe_kleiner", "gleich", "MeritMe_größer", "p", "r"])
bonf = ALPHA / max(len(tests), 1)
tests["signifikant_bonferroni"] = tests["p"] < bonf
print(f"Bonferroni-Schwelle: p < {bonf:.4f}")
tests.set_index(["variante", "methode"]).reindex(VARIANTS, level=0).round(3)

## 5. In welche Richtung? Werden Merkmale bevorzugt?

Der Rangabstand sagt, **wie weit** ein Zwilling rutscht, nicht **wohin**. Für Fairness entscheidend ist die Richtung:
Steht z. B. ein akademischer Zwilling systematisch besser da als sein Original?
`rang_gewinn` > 0 heißt: Der Zwilling steht weiter oben als das Original.

In [ ]:
richtung = (main[main["variante"] != "para"]
            .groupby(["variante", "richtung", "methode"])
            .agg(n=("rang_gewinn", "size"), Ø_rang_gewinn=("rang_gewinn", "mean"),
                 besser=("rang_gewinn", lambda x: (x > 0).mean()), schlechter=("rang_gewinn", lambda x: (x < 0).mean()))
            .round(2).unstack("methode"))
richtung

In [ ]:
plot = (main[main["variante"] != "para"].groupby(["variante", "richtung", "methode"])["rang_gewinn"].mean()
           .unstack("methode")[ALL_M])
fig, ax = plt.subplots(figsize=(9, 5.5))
y, off = np.arange(len(plot)), bar_offsets(len(ALL_M))
for i, m in enumerate(ALL_M):
    ax.barh(y + off[i], plot[m], height=0.8 / len(ALL_M), color=METHOD_COLORS[m], label=m)
ax.axvline(0, color=INK, lw=0.8)
ax.set_yticks(y, [f"{v} · {r}" for v, r in plot.index]); ax.invert_yaxis()
ax.set_xlabel("Ø Rangveränderung des Zwillings (Plätze; > 0 = Zwilling steht besser)")
ax.set_title("Richtung: Bevorzugt ein Merkmal den Zwilling?", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Gegenprobe zum Leck *computer science*:** Akademische edu-Zwillinge bekommen über „Bachelor's Degree in Computer
Science“ den ESCO-Skill *computer science*. Profitieren sie dort, wo der Job diesen Skill verlangt?

In [ ]:
from labels import _esco_skills
esco = _esco_skills()
cs_uri = esco.loc[esco["preferredLabel"] == "computer science", "conceptUri"].iloc[0]
acad = twins[(twins["variante"] == "edu") & (twins["richtung"] == "academic")][["twin_id", "assigned_job_id"]]
out = []
for method in [m for m in MERIT if m.startswith("hybrid")]:
    lab, _ = labels_for(method)
    a = acad.assign(job_will_cs=acad["assigned_job_id"].map(lambda j: cs_uri in lab[j]))
    a = a.merge(res[res["methode"] == method][["twin_id", "rang_gewinn", "delta_score"]], on="twin_id")
    out.append(a.groupby("job_will_cs").agg(n=("twin_id", "size"), Ø_rang_gewinn=("rang_gewinn", "mean"),
                                            Ø_delta_score=("delta_score", "mean")).assign(methode=method))
pd.concat(out).reset_index().set_index(["methode", "job_will_cs"]).round(3)

## Fazit

*(Zahlen aus den Ausgaben übernehmen; Abschnitt 2 und 4 aus dem ersten Lauf bereits eingetragen.)*

1. **Stufe 1:** Reines ESCOX verfehlt das Ziel beim realistischen Zwilling (all: 0,81) und schon beim reinen
   Umformulieren (para: 0,82). Der Hybrid liegt in allen Varianten bei ≥ 0,94.
2. **Proxy-Lecks:** ESCOX liest Spuren des Werdegangs als Skills (*demonstrate willingness to learn* bei allen
   all-Zwillingen, *teach computer science* bei edu, *manage professional transition in an arts career* bei lin).
   Der IT-Filter entfernt sie; übrig bleibt *computer science* über die Wortsuche (40 akademische edu-Zwillinge).
   *(Anteil im Job eintragen.)*
3. **Stufe 2:** MeritMe verschiebt die Ränge weniger als TF-IDF; mit Hybrid in allen Varianten signifikant.
   Beim reinen Umformulieren rutscht reines ESCOX aber stärker als TF-IDF (2,05 vs. 1,80 Plätze).
   TF-IDF liegt bei den Proxy-Varianten kaum über seiner Rauschgrenze (para).
4. **Richtung:** *(eintragen: Werden academic / confident / linear systematisch bevorzugt? Effekt von computer science?)*

**Einschränkungen:** TF-IDF-Baseline vorläufig; Rangabstand bei MeritMe überwiegend 0 durch gleichen Score
(Grobkörnigkeit, siehe Abschnitt 4); Skill-Begriffe sind per Prompt verankert, die Wortsuche ist daher per Konstruktion
stabil; nur Haupttest (A + B, je 100 Paare pro Variante).